# 센다 실험 — 단계 A 비교 실험 (코랩 GPU)
**이 노트북이 하는 일:** 저장소를 내려받아, 단계 A(개수 0~9)를 세 가지 설정 × 시드 3개로 학습합니다.

| 이름 | 설정 | 알아보려는 것 |
|---|---|---|
| **A4** | 실패 중심 출제만 (보너스 없음) | 환경만으로 합침이 풀리나 |
| **A5** | A2 보너스 + 실패 중심 출제 | 보너스에 얹으면 더 좋아지나 |
| **C0** | 보너스도 실패 중심도 없음 (대조군) | 위 둘의 효과를 분리해서 보기 위함 |

**처음 한 번만:** 런타임 → 런타임 유형 변경 → GPU(T4). 위에서부터 셀을 차례로 실행.
**끊겼을 때:** 같은 셀들을 처음부터 다시 실행하면 드라이브에 저장된 체크포인트(1000단계마다)부터 이어갑니다. 이미 끝난 시드는 건너뜁니다.

## 1. GPU 확인과 구글 드라이브 연결

In [ ]:
import torch
print('GPU 사용 가능:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/countworld_runs'      # 결과가 저장되는 폴더 (바꿔도 됨)
os.makedirs(BASE, exist_ok=True); print(BASE)

## 2. 저장소 내려받기

In [ ]:
import os, subprocess
REPO = 'gitdd12/integral_discovery'
BRANCH = 'ccr-206f948d-nkcq34'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    url = f'https://{token}@github.com/{REPO}.git'
except Exception:
    token = None
    url = f'https://github.com/{REPO}.git'
if not os.path.exists('/content/integral_discovery'):
    subprocess.run(['git','clone','-b',BRANCH,url,'/content/integral_discovery'], check=True)
else:
    subprocess.run(['git','-C','/content/integral_discovery','pull'], check=True)
%cd /content/integral_discovery
!git log --oneline | head -3

## 3. 설정 (필요하면 여기만 바꾸세요)

In [ ]:
SEEDS = [1, 2, 3]
STEPS = 12000          # T4 강화학습 단계 수 (단계 0은 자동으로 최대 4000단계)
MAX_PARALLEL = 3       # 동시에 돌릴 학습 수 (코랩 CPU가 2개라 3 이하 권장)
EXPERIMENTS = {
    'A4': dict(USAGE='0',   MI='0',   RAMP='0',   ENT='0.03', FOCUS='2'),
    'A5': dict(USAGE='1.0', MI='1.0', RAMP='0.3', ENT='0.03', FOCUS='2'),
    'C0': dict(USAGE='0',   MI='0',   RAMP='0',   ENT='0.03', FOCUS='0'),
}

## 4. 학습 실행 (끝날 때까지 이 셀이 돕니다. 1분마다 진행 상황을 출력)

In [ ]:
import os, subprocess, time, json, glob
jobs = []
for name, cfg in EXPERIMENTS.items():
    out = f'{BASE}/{name}'; os.makedirs(out, exist_ok=True)
    for s in SEEDS:
        if os.path.exists(f'{out}/s{s}.json'):
            continue
        env = dict(os.environ, OUT=out, LISTENERS='1', REFRESH='0', THREADS='1', DEVICE='cuda' if torch.cuda.is_available() else 'cpu', **cfg)
        jobs.append((name, s, env))
print('돌려야 할 학습:', len(jobs), '개')
running = []
def last_line(name, s):
    p = f'{BASE}/{name}/s{s}.log'
    if not os.path.exists(p): return '(시작 전)'
    L = [l for l in open(p).read().splitlines() if l.strip()]
    return L[-1][:100] if L else ''
t0 = time.time(); tick = 0
while jobs or running:
    running = [(n, s, p) for (n, s, p) in running if p.poll() is None]
    while jobs and len(running) < MAX_PARALLEL:
        n, s, env = jobs.pop(0)
        p = subprocess.Popen(['python', '-u', 'scripts/run_count_A.py', str(s), str(STEPS)], env=env,
                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        running.append((n, s, p))
    if tick % 6 == 0:
        print(f'--- {int((time.time()-t0)/60)}분 경과, 실행 중 {len(running)}개, 대기 {len(jobs)}개')
        for n, s, _ in running:
            print(f'  {n} 시드{s}:', last_line(n, s))
    tick += 1
    time.sleep(10)
print('모두 끝났습니다')

## 5. 결과 요약

In [ ]:
import json, glob, numpy as np
def summarize():
    for name in EXPERIMENTS:
        print(f'===== {name}')
        for s in SEEDS:
            p = f'{BASE}/{name}/s{s}.json'
            if not os.path.exists(p):
                print(f'  시드{s}: 아직'); continue
            r = json.load(open(p))['report']; e5 = r['e5']; adj = r['e4_adjacent']
            ok = sum(v >= 0.9 for v in adj.values())
            merged = {}
            for k, v in e5['modal'].items(): merged.setdefault(tuple(v), []).append(int(k))
            merged = [g for g in merged.values() if len(g) > 1]
            print(f"  시드{s}: 정확도 {r['e1_train']:.3f} | 빼둔쌍 {r['e1_heldout_pairs']:.3f} | n대n+1 ≥90% {ok}/9 | "
                  f"일관성 {e5['consistency']:.2f} 단사성 {e5['injective']:.2f} | 합쳐진 개수 {merged}")
summarize()

## 6. (선택) 결과를 저장소에 올리기
왼쪽 열쇠 아이콘(보안 비밀)에 `GITHUB_TOKEN`이 있고 쓰기 권한이 있을 때만 됩니다. 안 되면 5번 셀의 출력을 복사해서 대화에 붙여주셔도 됩니다.

In [ ]:
import shutil, subprocess
assert token, 'GITHUB_TOKEN 이 없습니다 (Secrets 에 추가하세요)'
dst = '/content/integral_discovery/runs3/colab'
for name in EXPERIMENTS:
    for f in glob.glob(f'{BASE}/{name}/s*.json') + glob.glob(f'{BASE}/{name}/s*.log'):
        os.makedirs(f'{dst}/{name}', exist_ok=True); shutil.copy(f, f'{dst}/{name}/')
cmds = [['git','config','user.email','colab@example.com'], ['git','config','user.name','colab'],
        ['git','add','runs3/colab'], ['git','commit','-m','코랩 단계 A 비교 실험 결과 (A4/A5/C0)'],
        ['git','push','origin',BRANCH]]
for c in cmds:
    print(subprocess.run(c, cwd='/content/integral_discovery', capture_output=True, text=True).stdout[-300:])